<a href="https://colab.research.google.com/github/Nahiya17/Technical-Documentation-Assistant/blob/main/GEN_AI_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TECHNICAL DOCUMENTATION

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
!pip install -q sentence-transformers transformers torch PyPDF2 faiss-cpu gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 29.2 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import PyPDF2
import faiss

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [4]:
from google.colab import files

uploaded = files.upload()

Saving Campus hiring 2027_DN 5.0_ DotNET_student brochure.pdf to Campus hiring 2027_DN 5.0_ DotNET_student brochure.pdf


In [5]:
print("Uploaded files:")

for filename in uploaded.keys():
    print(filename)

Uploaded files:
Campus hiring 2027_DN 5.0_ DotNET_student brochure.pdf


In [6]:
def extract_text_from_pdf(filename):

    pages = []

    pdf_reader = PyPDF2.PdfReader(filename)

    for page_number, page in enumerate(pdf_reader.pages):

        text = page.extract_text()

        if text and text.strip():

            pages.append({
                "text": text,
                "page": page_number + 1,
                "source": filename
            })

    return pages

In [7]:
all_pages = []

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        pages = extract_text_from_pdf(filename)

        all_pages.extend(pages)


print("Total pages extracted:", len(all_pages))

Total pages extracted: 6


In [8]:
def create_chunks(text, chunk_size=150, overlap=30):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [9]:
knowledge_base = []

for page in all_pages:

    chunks = create_chunks(
        page["text"],
        chunk_size=150,
        overlap=30
    )

    for chunk in chunks:

        knowledge_base.append({

            "text": chunk,

            "source": page["source"],

            "page": page["page"]

        })


print("Total chunks:", len(knowledge_base))

Total chunks: 12


In [10]:
print(knowledge_base[0])

{'text': 'Campus hiring 2027 Digital Nurture 5.0 (Dot NET FSE)', 'source': 'Campus hiring 2027_DN 5.0_ DotNET_student brochure.pdf', 'page': 1}


In [11]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
texts = [
    item["text"]
    for item in knowledge_base
]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (12, 384)


In [13]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print("Total vectors in FAISS:", index.ntotal)

Total vectors in FAISS: 12


In [14]:
def semantic_search(question, top_k=3):

    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    question_embedding = np.array(
        question_embedding
    ).astype("float32")

    scores, indices = index.search(
        question_embedding,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        results.append({

            "text": knowledge_base[idx]["text"],

            "source": knowledge_base[idx]["source"],

            "page": knowledge_base[idx]["page"],

            "score": float(score)

        })

    return results

In [15]:
question = "What is a Python list?"

results = semantic_search(question)

for result in results:

    print("\nSource:", result["source"])

    print("Page:", result["page"])

    print("Score:", round(result["score"], 3))

    print("Text:", result["text"][:500])


Source: Campus hiring 2027_DN 5.0_ DotNET_student brochure.pdf
Page: 5
Score: 0.209
Text: found in the name (refer to the naming convention matrix), uploaded scores, or submitted educational documents, the profile will be disqualified. Guidelines

Source: Campus hiring 2027_DN 5.0_ DotNET_student brochure.pdf
Page: 3
Score: 0.086
Text: Hiring process Note: If selected, you will join as a fresher as no prior work experience will be considered.If selectedAll assessment selects to appear for interview. Registration Screening Round 4 (Online) Technical interview Round 1 (Online) Communication assessment LOI (Letter of Intent)If cleared 3 weeks upskilling (self learning) Round 3 (Offline) Technical assessment 7 weeks deep skillingIf cleared Round 2 (Online) Qualifier assessment If cleared © 2026 -2027 Cognizant. All rights reserved

Source: Campus hiring 2027_DN 5.0_ DotNET_student brochure.pdf
Page: 6
Score: 0.069
Text: -Time Employment (FTE) onboarding.


In [18]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM',

In [20]:
def rag_answer(question):

    results = semantic_search(
        question,
        top_k=3
    )

    highest_score = results[0]["score"]

    # Grounding check

    if highest_score < 0.30:

        return {
            "answer":
            "Information not found in the knowledge base.",

            "sources": []
        }


    # Combine retrieved documents

    context = ""

    for result in results:

        context += (
            result["text"] +
            "\n\n"
        )


    # RAG prompt

    prompt = f"""
Answer the question using ONLY the
information provided in the context.

Do not use outside knowledge.

If the answer is not present in the
context, say:

Information not found in the knowledge base.

Context:

{context}

Question:

{question}

Answer:
"""


    # Generate answer

    response = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False
    )


    answer = response[0]["generated_text"]


    return {
        "answer": answer,
        "sources": results
    }

In [21]:
result = rag_answer(
    "tell about death zone?"
)

print("ANSWER:")
print(result["answer"])

ANSWER:
Information not found in the knowledge base.


In [22]:
print("\nDOCUMENTATION REFERENCES:")

for source in result["sources"]:

    print(
        f"\nSource: {source['source']}"
    )

    print(
        f"Page: {source['page']}"
    )

    print(
        f"Similarity Score: "
        f"{source['score']:.3f}"
    )


DOCUMENTATION REFERENCES:


In [23]:
result = rag_answer(
    "What is the capital of France?"
)

print(result["answer"])

Information not found in the knowledge base.


In [24]:
import gradio as gr

In [25]:
def chatbot(question):

    if not question.strip():

        return (
            "Please enter a question.",
            ""
        )


    result = rag_answer(question)


    answer = result["answer"]


    if not result["sources"]:

        return answer, "No relevant source found."


    references = "\n\n".join(

        [
            f"📄 {source['source']} "
            f"| Page {source['page']} "
            f"| Similarity: {source['score']:.3f}"

            for source in result["sources"]
        ]

    )


    return answer, references

In [26]:
demo = gr.Interface(

    fn=chatbot,

    inputs=gr.Textbox(
        label="Ask a question",
        placeholder=
        "Example: What is a Python list?"
    ),

    outputs=[
        gr.Textbox(
            label="🤖 Answer"
        ),

        gr.Textbox(
            label="📚 Documentation References"
        )
    ],

    title=
    "🤖 Technical Documentation Assistant",

    description=
    """
    A RAG-based chatbot that answers
    questions using only the uploaded
    technical documentation.
    """

)

In [27]:
demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36822ccc416ad8df2b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
